In [0]:
# Parameters - supplied by the Lakeflow Job via base_parameters
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_alias = dbutils.widgets.get("model_alias")
output_table = dbutils.widgets.get("output_table")

# Inter-task communication: get values from upstream tasks
full_model_name = dbutils.jobs.taskValues.get(taskKey="train_model", key="full_model_name")
full_input_table = dbutils.jobs.taskValues.get(taskKey="generate_data", key="full_table_name")
model_version = dbutils.jobs.taskValues.get(taskKey="train_model", key="model_version")

full_output_table = f"{catalog}.{schema}.{output_table}"

print(f"Model (from train_model task): {full_model_name}@{model_alias} (v{model_version})")
print(f"Input (from generate_data task): {full_input_table}")
print(f"Output: {full_output_table}")

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

# Load model using the parameterized alias
model = mlflow.pyfunc.load_model(f"models:/{full_model_name}@{model_alias}")
print(f"Loaded model: {full_model_name}@{model_alias}")

# Read input data
input_df = spark.read.table(full_input_table)
pdf = input_df.toPandas()

# Extract feature columns and run inference
feature_cols = [c for c in pdf.columns if c.startswith("feature_")]
features = pdf[feature_cols]
predictions = model.predict(features)

# Add predictions to the dataframe
pdf["prediction"] = predictions
print(f"Scored {len(pdf)} rows")

In [0]:
import json

# Convert back to Spark DataFrame and save
results_df = spark.createDataFrame(pdf)
results_df.write.format("delta").mode("overwrite").saveAsTable(full_output_table)

row_count = spark.table(full_output_table).count()
print(f"Successfully saved {row_count} predictions to {full_output_table}")

# Set task values (available if further downstream tasks are added)
dbutils.jobs.taskValues.set(key="rows_scored", value=row_count)
dbutils.jobs.taskValues.set(key="full_output_table", value=full_output_table)

# Exit with structured output for the orchestrator
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "rows_scored": row_count,
    "output_table": full_output_table,
    "model": full_model_name,
    "alias": model_alias
}))